In [0]:
%run ../../02_common_utils/operations

In [0]:
team_name="team_lemma"
catalog_name=f"charles_schwab_retailbrokerage_dev_{team_name}"
dbutils.widgets.text("batch_id","1","BATCH ID")
silver_watches=f"{catalog_name}.silver.watches"
gold_dim_customer=F"{catalog_name}.gold.dim_customer"
gold_dim_security=f"{catalog_name}.gold.dim_security"
gold_watches=f"{catalog_name}.gold.fact_watches"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

In [0]:
batch_id=dbutils.widgets.get("batch_id")

In [0]:
df_silver=spark.table(silver_watches)
df_gold_cust=spark.table(gold_dim_customer)
df_gold_sec=spark.table(gold_dim_security)

In [0]:
df_gold_sec=df_gold_sec.withColumn(
    "_batch",regexp_extract(col("_batch"), r'(\d+)', 1)
)


In [0]:
w = Window.partitionBy("W_C_ID", "W_S_SYMB", "W_ACTION").orderBy("W_DTS")
#placed date
df_actv = df_silver.filter(col("W_ACTION") == "ACTV") \
    .withColumn("pair_id", row_number().over(w)) \
    .select("W_C_ID", "W_S_SYMB", col("W_DTS").alias("DatePlaced"), "pair_id")
#removed date
df_cncl = df_silver.filter(col("W_ACTION") == "CNCL") \
    .withColumn("pair_id", row_number().over(w)) \
    .select("W_C_ID", "W_S_SYMB", col("W_DTS").alias("DateRemoved"), "pair_id")
df_events = df_actv.join(df_cncl, on=["W_C_ID", "W_S_SYMB", "pair_id"], how="left") \
    .drop("pair_id")

In [0]:
df_events.createOrReplaceTempView("df_events")
df_gold_cust.createOrReplaceTempView("df_gold_cust")
df_gold_sec.createOrReplaceTempView("df_gold_sec")

In [0]:

df_joined = spark.sql("""
    SELECT 
        w.W_C_ID, w.W_S_SYMB, w.DatePlaced, w.DateRemoved,
        c.sk_customerid, s.sk_securityid
    FROM df_events w
    INNER JOIN (
        SELECT customerid, sk_customerid,
               row_number() OVER (PARTITION BY customerid ORDER BY effectivedate DESC) AS rn
        FROM df_gold_cust
    ) c ON w.W_C_ID = c.customerid AND c.rn = 1
    INNER JOIN (
        SELECT symbol, sk_securityid,
               row_number() OVER (PARTITION BY symbol ORDER BY effectivedate DESC) AS rn
        FROM df_gold_sec
    ) s ON w.W_S_SYMB = s.symbol AND s.rn = 1
""")

# df_joined.limit(10).display()
# df_joined.count()

In [0]:
df_gold=df_joined.select(
    col("W_C_ID").alias("w_c_id"),
    col("W_S_SYMB").alias("w_s_symb"),
    col("sk_customerid"),
    col("sk_securityid"),
    expr("cast(date_format(DatePlaced,'yyyyMMdd') as bigint)").alias("sk_dateid_dateplaced"),
    expr("cast(date_format(DateRemoved,'yyyyMMdd') as bigint)").alias("sk_dateid_dateremoved"),
    lit(batch_id).alias("_batch"),
    current_timestamp().alias("_load_ts")
)
# df_gold.limit(10).display()
# df_gold.count()

run_id=df_silver.select("_run_id").first()[0]
df_gold=df_gold.withColumn("_run_id",lit(run_id))
df_gold.createOrReplaceTempView("df_gold")



In [0]:
try:
    #for batch 1 there will be no table so we need to create
    if spark.catalog.tableExists(gold_watches):
        print("Merging into existing table")
        spark.sql(f"""
                  merge into {gold_watches} as t 
                  using df_gold as s 
                  on t.w_c_id=s.w_c_id 
                  and t.w_s_symb=s.w_s_symb 
                  when matched then 
                  update set *
                  when not matched then
                  insert *
                  """)
        print("Merge successfully...")
        operation_type = "MERGE"
    else:
        print("creating gold table for first time")
        df_gold.write.format("delta").mode("overwrite").saveAsTable(gold_watches)
        print("Gold table created successfully....")
        operation_type = "OVERWRITE"
    
    # df_gold.limit(10).display()
    source_count=spark.read.table(silver_watches).count()
    gold_history=spark.sql(f"DESCRIBE HISTORY {gold_watches}").first()
    metrics=gold_history["operationMetrics"]

    if operation_type == "MERGE":
        inserted=int(metrics.get("numTargetRowsInserted", 0))
        updated=int(metrics.get("numTargetRowsUpdated", 0))
        deleted=int(metrics.get("numTargetRowsDeleted", 0))
        rows_affected=inserted+updated+deleted
    else: 
        # For Batch 1 OVERWRITE
        rows_affected=int(metrics.get("numOutputRows",0))

    target_count = spark.read.table(gold_watches).count()

    log_pipeline_recon(
        spark=spark,
        run_id=run_id,
        batch_id=batch_id,
        domain="CUSTOMER",
        table_name="fact_watches",
        source_layer="silver",       
        target_layer="gold",         
        source_count=source_count,
        target_count=target_count
    )
    
    log_audit_event(
        spark=spark,
        run_id=run_id,
        batch=batch_id,
        layer="gold",
        table_name="fact_watches",
        operation=operation_type,  
        rows_affected=rows_affected
    )

except Exception as e:
    raise e

In [0]:
# spark.table(gold_watches).count()